# Ordered Logistic Regression Results: FAIRˆ² Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ² dataset using the `mlcroissant` library. This dataset contains ordered logistic regression outputs and predictors for indigenous and modern knowledge adoption in rangeland management interventions in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (FAIRˆ² Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m\n")
print(metadata.description)

## 2. Data Overview
Review available **Record Sets**, **Fields**, and their `@id`s. All references below use the unique `@id` identifiers as defined by the Croissant schema and the dataset.

In [ ]:
# List all Record Sets and their Field/Column IDs
print('Available Record Sets and Fields:')
for record_set in dataset.record_sets:
    print(f"\n  Record Set: {record_set['@id']} (name: {record_set.get('name', '<no name>')})")
    if 'field' in record_set:
        fields = record_set['field']
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            print(f"    Field: {field['@id']} (name: {field.get('name', '<no name>')})")
            if 'column' in field:
                columns = field['column']
                if not isinstance(columns, list):
                    columns = [columns]
                for col in columns:
                    print(f"      Column: {col['@id']} (name: {col.get('name', '<no name>')})")

### Example: Preview Records via Their Record Set `@id`
Let's list some records for a specific record set using its `@id`. (Replace `<record_set_id>` with a real one from above as needed.)

In [ ]:
# List record set IDs found previously
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
if not record_set_ids:
    print('No record sets found in this dataset (Croissant schema may require a recordSet field).')
else:
    # Preview first 3 records from the first record set
    sample_record_set_id = record_set_ids[0]
    print(f'Previewing records for record set @id="{sample_record_set_id}":')
    for idx, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        if idx>=3:
            break
        print(record)

## 3. Data Extraction
Load data from *all* record sets into DataFrames for analysis. All entities are referenced by their `@id`s (record sets, fields, columns).

In [ ]:
# Extract data from each record set into a pandas DataFrame
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f'Loaded DataFrame for record set @id="{record_set_id}" with shape {df.shape}')

# Show example: the columns in the first available record set DataFrame
if record_set_ids:
    example_set = record_set_ids[0]
    print('Columns in record set @id="{}":'.format(example_set))
    print(dataframes[example_set].columns.tolist())
    dataframes[example_set].head()

## 4. Exploratory Data Analysis (EDA)

Apply standard data processing steps such as filtering, normalization, and grouping using only `@id` for fields.

In [ ]:
import numpy as np

# Example: Pick the first DataFrame/record set to demonstrate
if record_set_ids:
    eda_set_id = record_set_ids[0]
    df = dataframes[eda_set_id]
    
    # Try to find a numeric field (by looking for columns with numeric dtype, or guessing by name/column type)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to infer numeric columns by typical name hints
        for col in df.columns:
            if 'coeff' in col.lower() or 'value' in col.lower() or 'log' in col.lower() or 'score' in col.lower() or 'age' in col.lower():
                try:
                    df[col] = pd.to_numeric(df[col])
                    numeric_cols.append(col)
                except:
                    continue
    
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use first numeric column by @id/field name
        print(f'Example numeric field for EDA: {numeric_field_id}')
        
        # Filtering (remove NaNs first)
        df = df[df[numeric_field_id].notnull()]
        threshold = df[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {eda_set_id} where {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())
        
        # Normalizing that field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:\n", filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical field (by searching for object/string types)
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by {group_field_id}:")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped.head())
    else:
        print(f'No numeric fields found in record set @id="{eda_set_id}" for EDA.')

## 5. Visualization

Visualize distributions or relationships in the dataset using `matplotlib` (or `seaborn`). Illustrate results using field/column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: plot the distribution of the selected numeric field (if available)
if record_set_ids and numeric_cols:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, ax=ax, kde=True)
    ax.set_title(f'Distribution of {numeric_field_id} in record set @id={eda_set_id}')
    ax.set_xlabel(numeric_field_id)
    plt.show()

    # If grouped by a category
    if filtered_df.shape[0] and group_candidates:
        fig, ax = plt.subplots(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id, ax=ax)
        ax.set_title(f'{numeric_field_id} by {group_field_id} in record set @id={eda_set_id}')
        plt.xticks(rotation=40, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, process, and visualize statistical output datasets described via Croissant schemas using `mlcroissant`. All dataset structure, fields, and columns are referenced strictly by their `@id` unique identifiers for reproducibility. Use this notebook as a template for analyzing Croissant-packaged datasets in an auditable and schema-driven manner.